# attrition risk model (tabular ML, IBM HR Analytics dataset)

**Goal:** rank employees by how likely they are to leave, so HR can follow up with the people most at risk, and show *why* each person is flagged.

**Plan:** look at the data, split it fairly, compare a simple model (logistic regression) with a stronger one (random forest), test with cross-validation, tune one setting, choose a cutoff based on HR's capacity, explain individual scores, and finish with an interactive form.

# 1. Load the data
1,470 employees, one row each. The column `Attrition` says whether they left.

In [1]:
import os
import numpy as np
import pandas as pd

CSV = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
if not os.path.exists(CSV):
    import kagglehub
    path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
    CSV = os.path.join(path, CSV)

df = pd.read_csv(CSV)
print(df.shape)
df.head()

100%|██████████| 50.1k/50.1k [00:00<00:00, 16.6MB/s]

Extracting files...
(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 2. Why accuracy is the wrong score here
Only about 16% of employees left. A "model" that predicts *everyone stays* would be right about 84% of the time while catching **0 of the 237 leavers**. So I judge models with **recall** (of the people who really left, how many did I catch?), **precision** (of the people I flagged, how many really left?) and **PR-AUC** (how well the model ranks people, independent of any cutoff; a model with no skill scores about 0.16).

In [2]:
print(df["Attrition"].value_counts())
print(df["Attrition"].value_counts(normalize=True))

Attrition
No     1233
Yes     237
Name: count, dtype: int64
Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64


## 3. Train/test split
I hold back 20% of employees as a final exam the model never sees during training. `stratify=y` keeps the share of leavers (about 16%) the same in both parts, so the test set has enough leavers to measure recall. I also drop four columns that carry no information (constant values or an ID).

In [3]:
from sklearn.model_selection import train_test_split

df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])

y = (df["Attrition"] == "Yes").astype(int)   # 1 = left, 0 = stayed
X = df.drop(columns=["Attrition"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(len(X_train), len(X_test))
print(y_train.mean(), y_test.mean())

1176 294
0.16156462585034015 0.1598639455782313


## 4. Baseline and a stronger model
**Preprocessing:** words (like job role) become yes/no columns (one-hot encoding), and numbers are put on the same scale (standardisation).
**Baseline:** logistic regression with default settings. `class_weight="balanced"` makes missing a leaver count as a bigger mistake, so the model does not drift towards "everyone stays".
**Challenger:** random forest (many decision trees that vote).

In [4]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, average_precision_score

cat_cols = X_train.select_dtypes(include="object").columns
num_cols = X_train.select_dtypes(exclude="object").columns

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

baseline = Pipeline([
    ("prep", clone(prep)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
rf = Pipeline([
    ("prep", clone(prep)),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                   class_weight="balanced", random_state=42)),
])
baseline.fit(X_train, y_train)
rf.fit(X_train, y_train)

for name, m in [("Logistic regression", baseline), ("Random forest", rf)]:
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    print(name,
          "| precision:", round(precision_score(y_test, pred), 2),
          "| recall:", round(recall_score(y_test, pred), 2),
          "| PR-AUC:", round(average_precision_score(y_test, proba), 2))

Logistic regression | precision: 0.35 | recall: 0.64 | PR-AUC: 0.56
Random forest | precision: 0.5 | recall: 0.28 | PR-AUC: 0.44


## 5. Cross-validation (a fairer comparison)
The test set holds only about 47 leavers, so one split can be lucky or unlucky. Cross-validation trains and tests 5 times on different splits and reports the average and the spread (±).

**What I found:** the two models rank employees about equally well (PR-AUC 0.59 ± 0.05 vs 0.57 ± 0.07, a difference smaller than the spread). They differ in where they draw the line: at the default cutoff the logistic regression catches far more leavers (recall 0.71 vs 0.32), the random forest is right more often when it flags someone (precision 0.71 vs 0.36). I chose logistic regression because it ranks as well, is simpler, and its scores can be explained per person.

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["precision", "recall", "average_precision"]

for name, m in [("Logistic regression", baseline), ("Random forest", rf)]:
    res = cross_validate(m, X, y, cv=cv, scoring=scoring)
    print(name)
    for s in scoring:
        v = res["test_" + s]
        print(f"  {s}: {v.mean():.2f} ± {v.std():.2f}")

Logistic regression
  precision: 0.36 ± 0.02
  recall: 0.71 ± 0.04
  average_precision: 0.59 ± 0.05
Random forest
  precision: 0.71 ± 0.13
  recall: 0.32 ± 0.02
  average_precision: 0.57 ± 0.07


## 6. Tuning the regularisation strength (C)
`C` controls how simple the logistic regression stays: a small C shrinks the weights and stops the model from memorising noise in a small dataset. I tested four values with the same cross-validation.

**What I found:** C = 0.1 scored best (0.613), but the gap to the default C = 1 (0.589) is smaller than the spread (about ±0.05), so this is a small, not a proven, gain. I use C = 0.1 because a simpler model is the safer choice on 1,470 rows and did at least as well.

In [6]:
for C in [0.01, 0.1, 1, 10]:
    m = Pipeline([
        ("prep", clone(prep)),
        ("clf", LogisticRegression(C=C, max_iter=1000, class_weight="balanced")),
    ])
    res = cross_validate(m, X, y, cv=cv, scoring=["average_precision"])
    v = res["test_average_precision"]
    print(f"C={C}: PR-AUC {v.mean():.3f} ± {v.std():.3f}")

C=0.01: PR-AUC 0.610 ± 0.039
C=0.1: PR-AUC 0.613 ± 0.049
C=1: PR-AUC 0.589 ± 0.047
C=10: PR-AUC 0.583 ± 0.049


## 7. Final model
Logistic regression with C = 0.1, trained on the training set.

In [7]:
model = Pipeline([
    ("prep", clone(prep)),
    ("clf", LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)
print("C used:", model.named_steps["clf"].C)

C used: 0.1


## 8. Choosing the cutoff
The cutoff decides who gets flagged, so it is a business decision. I assumed HR can hold follow-up conversations with about 150 people (roughly 10% of staff). To avoid judging scores on people the model trained on, every employee below is scored by a model that did not see them (`cross_val_predict`).

**What I found:** a cutoff of **0.76 flags 152 people, with precision 0.68 and recall 0.43**. That means about 103 of the 237 leavers are caught, and random guessing would give only about 16% precision. If HR's capacity changes, the cutoff moves with it (for example about 470 people at 0.5, with recall 0.73 but only 37% precision).

In [8]:
from sklearn.model_selection import cross_val_predict

proba_cv = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]

for t in [0.3, 0.5, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.9, 0.95]:
    p = (proba_cv >= t).astype(int)
    print(f"cutoff {t}: flagged {p.sum():4d} | precision {precision_score(y, p):.2f} | recall {recall_score(y, p):.2f}")

cutoff 0.3: flagged  779 | precision 0.27 | recall 0.88
cutoff 0.5: flagged  470 | precision 0.37 | recall 0.73
cutoff 0.7: flagged  216 | precision 0.57 | recall 0.52
cutoff 0.72: flagged  190 | precision 0.61 | recall 0.49
cutoff 0.74: flagged  173 | precision 0.64 | recall 0.47
cutoff 0.76: flagged  152 | precision 0.68 | recall 0.43
cutoff 0.78: flagged  136 | precision 0.74 | recall 0.42
cutoff 0.8: flagged  118 | precision 0.76 | recall 0.38
cutoff 0.9: flagged   39 | precision 0.90 | recall 0.15
cutoff 0.95: flagged    8 | precision 1.00 | recall 0.03


## 9. Explaining one employee's score
For logistic regression, a feature's push on one person's score = the model's weight for that feature × that person's own (scaled) value. Positive pushes risk up, negative pushes it down. Below: the highest-risk employee in the test set and their top 5 reasons.

In [9]:
names = model.named_steps["prep"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

Xt = model.named_steps["prep"].transform(X_test)
Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt

proba = model.predict_proba(X_test)[:, 1]
i = int(np.argmax(proba))

contrib = pd.Series(Xt[i] * coefs, index=names)
top = contrib.sort_values(key=abs, ascending=False).head(5)

print("Risk score:", round(proba[i], 2))
print(X_test.iloc[i][["JobRole", "Age", "OverTime", "BusinessTravel"]])
print(top.round(2))

Risk score: 0.98
JobRole           Sales Representative
Age                                 21
OverTime                           Yes
BusinessTravel       Travel_Frequently
Name: 357, dtype: object
cat__OverTime_Yes                        0.69
cat__BusinessTravel_Travel_Frequently    0.58
num__EnvironmentSatisfaction             0.58
num__Age                                 0.49
cat__JobRole_Sales Representative        0.44
dtype: float64


## 10. Interactive demo
Set an employee's details and press the button. The form sets 6 fields; **all other fields are filled with the most common value (words) or the median (numbers) from the training data**. The flag uses the cutoff of 0.76 chosen above. The score is a ranking score, not a true probability.

In [10]:
import ipywidgets as widgets
from IPython.display import display

CUTOFF = 0.76
typical = {c: (X_train[c].mode()[0] if c in cat_cols else X_train[c].median())
           for c in X_train.columns}

role = widgets.Dropdown(options=sorted(X_train["JobRole"].unique()), description="Job role")
overtime = widgets.Dropdown(options=["Yes", "No"], description="Overtime")
travel = widgets.Dropdown(options=sorted(X_train["BusinessTravel"].unique()), description="Travel")
age = widgets.IntSlider(value=35, min=18, max=60, description="Age")
env = widgets.IntSlider(value=3, min=1, max=4, description="Env. satisf.")
years = widgets.IntSlider(value=10, min=0, max=40, description="Work years")
button = widgets.Button(description="Predict risk")
out = widgets.Output()

def on_click(_):
    row = pd.DataFrame([typical])
    row["JobRole"] = role.value
    row["OverTime"] = overtime.value
    row["BusinessTravel"] = travel.value
    row["Age"] = age.value
    row["EnvironmentSatisfaction"] = env.value
    row["TotalWorkingYears"] = years.value
    p = model.predict_proba(row)[0, 1]
    xt = model.named_steps["prep"].transform(row)
    xt = xt.toarray() if hasattr(xt, "toarray") else xt
    top = pd.Series(xt[0] * coefs, index=names).sort_values(key=abs, ascending=False).head(3)
    with out:
        out.clear_output()
        print("Risk score:", round(p, 2))
        print("FLAGGED for HR follow-up" if p >= CUTOFF else "Not flagged")
        print(f"(cutoff {CUTOFF}: a ranking score, not a true probability)")
        print(top.round(2))

button.on_click(on_click)
display(role, overtime, travel, age, env, years, button, out)

Dropdown(description='Job role', options=('Healthcare Representative', 'Human Resources', 'Laboratory Technici…

Dropdown(description='Overtime', options=('Yes', 'No'), value='Yes')

Dropdown(description='Travel', options=('Non-Travel', 'Travel_Frequently', 'Travel_Rarely'), value='Non-Travel…

IntSlider(value=35, description='Age', max=60, min=18)

IntSlider(value=3, description='Env. satisf.', max=4, min=1)

IntSlider(value=10, description='Work years', max=40)

Button(description='Predict risk', style=ButtonStyle())

Output()

## 11. Limitations

1. **Synthetic data.** The IBM HR dataset is artificial and meant for practice. Features that stand out here (like Sales Reps leaving more often) might not hold true in an actual company environment.
2. **Small data.** There are only have 237 employees who actually left. Because of the small sample size, cross-validation metrics fluctuate a bit from fold to fold (hence the $\pm$ values in the results).
3. **Scores are rankings, not probabilities.** `class_weight="balanced"` stretches the scores, so 0.88 does not mean an 88% chance of leaving.
4. **Correlation, not cause.** The model picks up on statistical patterns (like overtime being linked to turnover), but that doesn't mean cutting overtime will automatically keep someone from leaving.
5. **Optimistic numbers.**  I tuned hyperparameter $C$ and tested cutoffs on the same cross-validation loop I used to report performance. Running a nested cross-validation would give a strictly unbiased score.
6. **Default values in the demo:** The interactive widget only takes 6 user inputs and fills the remaining features with dataset medians or modes. This pulls predictions toward the average and might understate risk for edge cases.
7. **Fairness.** Age and other personal fields are inputs. Using a model like this for real decisions about people needs care and human review.